# BCS Determination — Test des modeles sur les images Reddit + Detection de contours

Ce notebook permet de tester **tous les modeles entraines** sur les images Reddit :
- **Classification** : ResNet50 (acc=0.79) et ViT-B/16 (acc=0.87) sur Stanford Dogs (120 races)
- **Segmentation** : DeepLabV3-ResNet50 (IoU=0.82) sur Oxford-IIIT Pet (3 classes)
- **Detection de contours classique** : Canny, Sobel, Laplacien (OpenCV + scikit-image)

Les images Reddit proviennent du post *"Is my dog overweight?"* et sont hors distribution.

---
## 1. Configuration & imports

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/home/SPeillet/bcs_determination")
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from PIL import Image
from glob import glob

import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.datasets import ImageFolder

import cv2
import skimage
from skimage import filters as skfilters
from skimage import feature as skfeature
from skimage import color as skcolor

from bcs_pipeline.lightning_module.classification_module import LitClassificationModule
from bcs_pipeline.lightning_module.segmentation_module import (
    LitSegmentationModule,
    transforms_inv_normalize,
    overlay_mask_on_image,
)

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device     : {device}")
if device.type == "cuda":
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
print(f"OpenCV     : {cv2.__version__}")
print(f"scikit-image: {skimage.__version__}")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

RESNET_CKPT = (
    ROOT / "experiments/resnet50_adam_cosine_annealing/checkpoints"
    / "epoch=epoch=15-val_acc=val_acc=0.79-step=8240.ckpt"
)
VIT_CKPT = (
    ROOT / "experiments/vit_adam_cosine_annealing/checkpoints"
    / "epoch=epoch=21-val_acc=val/acc=0.87-step=11330.ckpt"
)
SEG_CKPT = (
    ROOT / "experiments/deeplabv3_resnet50_adam_cosine_annealing/checkpoints"
    / "epoch=epoch=30-val_iou=val/iou=0.82-step=6417.ckpt"
)
DATA_DIR = ROOT / "data" / "Stanford_dogs"
REDDIT_DIR = ROOT / "data" / "Reddit_example"

IMAGE_SIZE_CLF = 224
IMAGE_SIZE_SEG = 256
NUM_CLASSES_CLF = 120
NUM_CLASSES_SEG = 3
SEG_CLASS_NAMES = ["Pet (foreground)", "Background", "Border"]
TOP_K = 5

for name, path in [("ResNet50", RESNET_CKPT), ("ViT", VIT_CKPT), ("DeepLabV3", SEG_CKPT)]:
    print(f"{name:10s} checkpoint: {path.name} (exists={path.exists()})")

In [ ]:
reddit_images = sorted(REDDIT_DIR.glob("*.webp"))
print(f"Images Reddit trouvees : {len(reddit_images)}")
for p in reddit_images:
    print(f"  -> {p.name}")

In [ ]:
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

clf_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE_CLF * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE_CLF),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

seg_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE_SEG, IMAGE_SIZE_SEG)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

full_dataset = ImageFolder(root=str(DATA_DIR / "Images"))
class_names_raw = full_dataset.classes
class_names = ["-".join(c.split("-")[1:]) for c in class_names_raw]
print(f"Nombre de classes : {len(class_names)}")
print(f"Exemples          : {class_names[:5]}")

In [ ]:
fig, axes = plt.subplots(1, len(reddit_images), figsize=(7 * len(reddit_images), 7))
if len(reddit_images) == 1:
    axes = [axes]
for ax, img_path in zip(axes, reddit_images):
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=10)
    ax.axis("off")
fig.suptitle("Images Reddit - Is my dog overweight?", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 2. Chargement des modeles

In [ ]:
print("Chargement ResNet50...")
checkpoint = torch.load(str(RESNET_CKPT), map_location=device, weights_only=False)
ckpt_state = checkpoint["state_dict"]

has_sequential_fc = any("net.backbone.fc.1" in k for k in ckpt_state.keys())
dropout_val = 0.3 if has_sequential_fc else 0.0

resnet_model = LitClassificationModule(
    model_name="resnet50",
    num_classes=NUM_CLASSES_CLF,
    regularization={"dropout": dropout_val, "stochastic_depth": 0.0},
)

model_state = resnet_model.state_dict()
new_state = {}
for key, val in ckpt_state.items():
    if key in model_state:
        new_state[key] = val
    elif key == "net.backbone.fc.weight":
        new_state["net.backbone.fc.1.weight"] = val
    elif key == "net.backbone.fc.bias":
        new_state["net.backbone.fc.1.bias"] = val
    else:
        new_state[key] = val

resnet_model.load_state_dict(new_state, strict=False)
resnet_model.to(device).eval()
print(f"  ResNet50 charge : {sum(p.numel() for p in resnet_model.parameters()):,} parametres")

In [ ]:
print("Chargement ViT...")
vit_model = LitClassificationModule.load_from_checkpoint(
    str(VIT_CKPT),
    map_location=device,
)
vit_model.to(device).eval()
print(f"  ViT charge      : {sum(p.numel() for p in vit_model.parameters()):,} parametres")

In [ ]:
print("Chargement DeepLabV3-ResNet50...")
seg_model = LitSegmentationModule.load_from_checkpoint(
    str(SEG_CKPT),
    map_location="cpu",
)
seg_model.eval().to(device).freeze()
print(f"  DeepLabV3 charge : {sum(p.numel() for p in seg_model.parameters()):,} parametres")

---
## 3. Classification — ResNet50 vs ViT

In [ ]:
def predict_topk(model, img_pil, transform, top_k=5):
    img_tensor = transform(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(img_tensor)
        probs = F.softmax(logits, dim=1)
        top_probs, top_indices = probs.topk(top_k, dim=1)
    return top_probs[0].cpu().numpy(), top_indices[0].cpu().numpy()

fig, axes = plt.subplots(len(reddit_images), 3, figsize=(20, 6 * len(reddit_images)))
if len(reddit_images) == 1:
    axes = axes[np.newaxis, :]

for row, img_path in enumerate(reddit_images):
    img_pil = Image.open(img_path).convert("RGB")

    resnet_probs, resnet_idx = predict_topk(resnet_model, img_pil, clf_transform, TOP_K)
    vit_probs, vit_idx = predict_topk(vit_model, img_pil, clf_transform, TOP_K)

    img_vis = denormalize(clf_transform(img_pil))
    axes[row, 0].imshow(img_vis.permute(1, 2, 0).numpy())
    axes[row, 0].set_title(img_path.name, fontsize=11)
    axes[row, 0].axis("off")

    for col, (probs, indices, title, color) in enumerate([
        (resnet_probs, resnet_idx, f"ResNet50 (Top-{TOP_K})", "#2196F3"),
        (vit_probs, vit_idx, f"ViT-B/16 (Top-{TOP_K})", "#FF9800"),
    ], start=1):
        pred_names = [class_names[indices[i]] for i in range(TOP_K)]
        pred_confs = [probs[i] for i in range(TOP_K)]
        colors = ["#4CAF50" if i == 0 else color for i in range(TOP_K)]

        bars = axes[row, col].barh(range(TOP_K - 1, -1, -1), pred_confs, color=colors, edgecolor="black")
        axes[row, col].set_yticks(range(TOP_K - 1, -1, -1))
        axes[row, col].set_yticklabels(pred_names, fontsize=9)
        axes[row, col].set_xlabel("Probabilite")
        axes[row, col].set_title(title, fontsize=11)
        axes[row, col].set_xlim(0, 1)

        for bar, conf in zip(bars, pred_confs):
            axes[row, col].text(
                bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{conf:.2%}", va="center", fontsize=8,
            )

    print(f"\n{img_path.name}:")
    print(f"  ResNet50 : {class_names[resnet_idx[0]]} ({resnet_probs[0]:.2%})")
    print(f"  ViT      : {class_names[vit_idx[0]]} ({vit_probs[0]:.2%})")

fig.suptitle("Classification ResNet50 vs ViT — Images Reddit", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 3.1 Comparaison des probabilites ResNet50 vs ViT

In [ ]:
fig, axes = plt.subplots(len(reddit_images), 2, figsize=(14, 5 * len(reddit_images)))
if len(reddit_images) == 1:
    axes = axes[np.newaxis, :]

for row, img_path in enumerate(reddit_images):
    img_pil = Image.open(img_path).convert("RGB")

    with torch.no_grad():
        resnet_logits = resnet_model(clf_transform(img_pil).unsqueeze(0).to(device))
        resnet_all = F.softmax(resnet_logits, dim=1).cpu().numpy().flatten()

        vit_logits = vit_model(clf_transform(img_pil).unsqueeze(0).to(device))
        vit_all = F.softmax(vit_logits, dim=1).cpu().numpy().flatten()

    top5_r = np.argsort(resnet_all)[::-1][:5]
    top5_v = np.argsort(vit_all)[::-1][:5]
    all_top = list(dict.fromkeys(list(top5_r) + list(top5_v)))

    x = np.arange(len(all_top))
    width = 0.35

    r_vals = [resnet_all[i] for i in all_top]
    v_vals = [vit_all[i] for i in all_top]
    labels = [class_names[i] for i in all_top]

    axes[row, 0].barh(x + width / 2, r_vals, width, label="ResNet50", color="#2196F3", edgecolor="black")
    axes[row, 0].barh(x - width / 2, v_vals, width, label="ViT", color="#FF9800", edgecolor="black")
    axes[row, 0].set_yticks(x)
    axes[row, 0].set_yticklabels(labels, fontsize=8)
    axes[row, 0].set_xlabel("Probabilite")
    axes[row, 0].set_title(f"Top classes — {img_path.name[:30]}")
    axes[row, 0].legend()

    agree = set(top5_r) & set(top5_v)
    labels_pie = ["Classes communes Top-5", "ResNet50 seul", "ViT seul"]
    sizes = [len(agree), len(set(top5_r) - agree), len(set(top5_v) - agree)]
    colors_pie = ["#4CAF50", "#2196F3", "#FF9800"]
    axes[row, 1].pie([s for s in sizes if s > 0],
                     labels=[l for l, s in zip(labels_pie, sizes) if s > 0],
                     colors=[c for c, s in zip(colors_pie, sizes) if s > 0],
                     autopct="%1.0f%%", startangle=90)
    axes[row, 1].set_title("Concordance Top-5")

plt.tight_layout()
plt.show()

---
## 4. Segmentation — DeepLabV3-ResNet50

In [ ]:
PALETTE = [(0, 0, 0), (255, 255, 255), (128, 128, 128)]

def mask_to_rgb(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_idx, color in enumerate(PALETTE):
        rgb[mask == cls_idx] = color
    return rgb

inv_normalize = transforms_inv_normalize()

fig, axes = plt.subplots(len(reddit_images), 4, figsize=(18, 5 * len(reddit_images)))
if len(reddit_images) == 1:
    axes = axes[np.newaxis, :]

col_titles = ["Image originale", "Masque predit", "Overlay", "Repartition pixels"]
for col, t in enumerate(col_titles):
    axes[0, col].set_title(t, fontsize=13, fontweight="bold")

with torch.no_grad():
    for row, img_path in enumerate(reddit_images):
        img_pil = Image.open(img_path).convert("RGB")
        img_tensor = seg_transform(img_pil).unsqueeze(0).to(device)

        logits = seg_model(img_tensor)
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu()

        img_vis = inv_normalize(seg_transform(img_pil))

        axes[row, 0].imshow(img_vis.permute(1, 2, 0).numpy())
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask_to_rgb(pred.numpy()))
        axes[row, 1].axis("off")

        overlay = overlay_mask_on_image(img_vis, pred, alpha=0.4, palette=PALETTE)
        axes[row, 2].imshow(overlay.numpy().transpose(1, 2, 0))
        axes[row, 2].axis("off")

        class_pixels = [(pred == c).sum().item() for c in range(NUM_CLASSES_SEG)]
        total_px = pred.numel()
        pcts = [100 * cp / total_px for cp in class_pixels]
        colors_bar = ["#333333", "#DDDDDD", "#888888"]
        bars = axes[row, 3].barh(range(NUM_CLASSES_SEG), pcts, color=colors_bar, edgecolor="black")
        axes[row, 3].set_yticks(range(NUM_CLASSES_SEG))
        axes[row, 3].set_yticklabels(SEG_CLASS_NAMES, fontsize=9)
        axes[row, 3].set_xlabel("Pourcentage (%)")
        for bar, pct in zip(bars, pcts):
            axes[row, 3].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                              f"{pct:.1f}%", va="center", fontsize=9)

        axes[row, 0].set_ylabel(img_path.name[:30], fontsize=9, rotation=0, labelpad=120)
        print(f"\n{img_path.name}:")
        for c in range(NUM_CLASSES_SEG):
            print(f"  {SEG_CLASS_NAMES[c]:25s}: {pcts[c]:.1f}%")

plt.tight_layout()
plt.show()

---
## 5. Detection de contours classique (OpenCV + scikit-image)

Application d'algorithmes classiques de vision par ordinateur (sans deep learning)
sur les images Reddit : Canny, Sobel, Laplacien, Prewitt, Roberts.

In [ ]:
methods = {
    "Original (gris)": None,
    "Canny (50-100)":  lambda g: cv2.Canny(g, 50, 100),
    "Canny (100-200)": lambda g: cv2.Canny(g, 100, 200),
    "Canny (150-300)": lambda g: cv2.Canny(g, 150, 300),
    "Sobel magnitude": lambda g: np.sqrt(cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3)**2
                                       + cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3)**2),
    "Laplacien (CV2)": lambda g: cv2.Laplacian(g, cv2.CV_64F),
    "Canny+Blur":      lambda g: cv2.Canny(cv2.GaussianBlur(g, (5, 5), 0), 100, 200),
    "Prewitt (sk)":    lambda g: (skfilters.prewitt(g.astype(float) / 255.0) * 255).astype(np.uint8),
    "Roberts (sk)":    lambda g: (skfilters.roberts(g.astype(float) / 255.0) * 255).astype(np.uint8),
    "Scharr (sk)":     lambda g: (skfilters.scharr(g.astype(float) / 255.0) * 255).astype(np.uint8),
}

for img_path in reddit_images:
    img_pil = Image.open(img_path).convert("RGB")
    img_np = np.array(img_pil)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    n = len(methods)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_flat = axes.flatten()

    for i, (name, fn) in enumerate(methods.items()):
        if fn is None:
            axes_flat[i].imshow(gray, cmap="gray")
        else:
            result = fn(gray)
            axes_flat[i].imshow(result, cmap="gray")
        axes_flat[i].set_title(name, fontsize=10)
        axes_flat[i].axis("off")

    for j in range(n, len(axes_flat)):
        axes_flat[j].axis("off")

    fig.suptitle(f"Detection de contours — {img_path.name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

### 5.1 Multi-scale Canny (scikit-image)

In [ ]:
sigmas = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]

for img_path in reddit_images:
    img_pil = Image.open(img_path).convert("RGB")
    img_np = np.array(img_pil)
    gray_float = skcolor.rgb2gray(img_np)

    fig, axes = plt.subplots(1, len(sigmas) + 1, figsize=(4 * (len(sigmas) + 1), 4))
    axes[0].imshow(gray_float, cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")

    for i, sigma in enumerate(sigmas, 1):
        edges = skfeature.canny(gray_float, sigma=sigma)
        axes[i].imshow(edges, cmap="gray")
        axes[i].set_title(f"Canny \u03c3={sigma}")
        axes[i].axis("off")

    fig.suptitle(f"Multi-scale Canny — {img_path.name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

### 5.2 Overlay : contours superposes a l'image originale

In [ ]:
def overlay_edges(rgb, edges_gray, color=(0, 255, 0), alpha=0.6):
    overlay = rgb.copy()
    mask = edges_gray > 0
    if mask.dtype != bool:
        mask = mask > (edges_gray.max() * 0.3) if edges_gray.max() > 0 else mask > 0
    overlay[mask] = ((1 - alpha) * rgb[mask] + alpha * np.array(color)).astype(np.uint8)
    return overlay

fig, axes = plt.subplots(len(reddit_images), 3, figsize=(18, 6 * len(reddit_images)))
if len(reddit_images) == 1:
    axes = axes[np.newaxis, :]

for row, img_path in enumerate(reddit_images):
    img_pil = Image.open(img_path).convert("RGB")
    rgb = np.array(img_pil)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    gray_float = skcolor.rgb2gray(rgb)

    canny_cv = cv2.Canny(gray, 100, 200)
    canny_blur = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 100, 200)
    canny_sk = (skfeature.canny(gray_float, sigma=2.0).astype(np.float64) * 255)

    axes[row, 0].imshow(overlay_edges(rgb, canny_cv, color=(0, 255, 0)))
    axes[row, 0].set_title("Canny OpenCV (100-200)")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(overlay_edges(rgb, canny_blur, color=(255, 100, 0)))
    axes[row, 1].set_title("Canny+Blur OpenCV")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(overlay_edges(rgb, canny_sk, color=(0, 150, 255)))
    axes[row, 2].set_title("Canny scikit-image (\u03c3=2)")
    axes[row, 2].axis("off")

fig.suptitle("Contours superposes — Images Reddit", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Vue d'ensemble : Classification + Segmentation + Contours

In [ ]:
for img_path in reddit_images:
    img_pil = Image.open(img_path).convert("RGB")
    img_np = np.array(img_pil)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    resnet_probs, resnet_idx = predict_topk(resnet_model, img_pil, clf_transform, TOP_K)
    vit_probs, vit_idx = predict_topk(vit_model, img_pil, clf_transform, TOP_K)

    with torch.no_grad():
        seg_tensor = seg_transform(img_pil).unsqueeze(0).to(device)
        seg_logits = seg_model(seg_tensor)
        seg_pred = torch.argmax(seg_logits, dim=1).squeeze(0).cpu()
    seg_overlay = overlay_mask_on_image(
        inv_normalize(seg_transform(img_pil)), seg_pred, alpha=0.4, palette=PALETTE
    )

    canny_result = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 100, 200)
    canny_overlay = overlay_edges(img_np, canny_result, color=(0, 255, 0), alpha=0.6)

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    axes[0, 0].imshow(img_np)
    axes[0, 0].set_title("Image originale", fontsize=12, fontweight="bold")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(seg_overlay.numpy().transpose(1, 2, 0))
    axes[0, 1].set_title("Segmentation DeepLabV3", fontsize=12, fontweight="bold")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(canny_overlay)
    axes[0, 2].set_title("Contours Canny", fontsize=12, fontweight="bold")
    axes[0, 2].axis("off")

    for col, (probs, indices, title, color) in enumerate([
        (resnet_probs, resnet_idx, f"ResNet50 Top-{TOP_K}", "#2196F3"),
        (vit_probs, vit_idx, f"ViT-B/16 Top-{TOP_K}", "#FF9800"),
    ], start=0):
        pred_names = [class_names[indices[i]] for i in range(TOP_K)]
        pred_confs = [probs[i] for i in range(TOP_K)]
        colors = ["#4CAF50" if i == 0 else color for i in range(TOP_K)]
        bars = axes[1, col].barh(range(TOP_K - 1, -1, -1), pred_confs, color=colors, edgecolor="black")
        axes[1, col].set_yticks(range(TOP_K - 1, -1, -1))
        axes[1, col].set_yticklabels(pred_names, fontsize=8)
        axes[1, col].set_xlim(0, 1)
        axes[1, col].set_title(title, fontsize=12, fontweight="bold")
        for bar, conf in zip(bars, pred_confs):
            axes[1, col].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                              f"{conf:.2%}", va="center", fontsize=8)

    class_pixels = [(seg_pred == c).sum().item() for c in range(NUM_CLASSES_SEG)]
    total_px = seg_pred.numel()
    pcts = [100 * cp / total_px for cp in class_pixels]
    colors_bar = ["#333333", "#DDDDDD", "#888888"]
    bars = axes[1, 2].barh(range(NUM_CLASSES_SEG), pcts, color=colors_bar, edgecolor="black")
    axes[1, 2].set_yticks(range(NUM_CLASSES_SEG))
    axes[1, 2].set_yticklabels(SEG_CLASS_NAMES, fontsize=9)
    axes[1, 2].set_title("Segmentation (% pixels)", fontsize=12, fontweight="bold")
    for bar, pct in zip(bars, pcts):
        axes[1, 2].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                        f"{pct:.1f}%", va="center", fontsize=9)

    fig.suptitle(f"Vue d'ensemble — {img_path.name}", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

---
## 7. Resume

| Modele | Tache | Dataset d'entrainement | Performance |
|--------|-------|----------------------|-------------|
| **ResNet50** | Classification (120 races) | Stanford Dogs | val_acc=0.79 |
| **ViT-B/16** | Classification (120 races) | Stanford Dogs | val_acc=0.87 |
| **DeepLabV3-ResNet50** | Segmentation (3 classes) | Oxford-IIIT Pet | val_iou=0.82 |

Analyse visuelle sur les images Reddit :
- **Classification** : comparer les predictions ResNet50 vs ViT, analyser la confiance
- **Segmentation** : evaluer la separation pet/background/border
- **Contours classiques** : Canny, Sobel, Laplacien comme baseline sans apprentissage